In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-10-02 21:27:11 +05:30)


In [2]:
import tensorflow as tf

import os
from pathlib import Path

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

import numpy as np

time: 8.91 s (started: 2026-10-02 21:27:11 +05:30)


In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" 

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

MODELS_DIR = PROJECT_ROOT / "models" / "phase1"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists(), MODELS_DIR.exists())

True True True True
time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


In [6]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


In [7]:
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


In [8]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale'
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale'
)

Found 4693 images belonging to 2 classes.
Found 539 images belonging to 2 classes.
time: 344 ms (started: 2026-10-02 21:27:20 +05:30)


In [9]:
labels = train_generator.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))

print("Class indices:", train_generator.class_indices)
print("Class distribution:", np.bincount(labels))
print("Original Class weights:", class_weights)

class_weights = {
    0: 1.6,   # NORMAL (was ~1.95)
    1: 0.8    # PNEUMONIA (was ~0.67)
}

print("Adjusted Class weights:", class_weights)

Class indices: {'NORMAL': 0, 'PNEUMONIA': 1}
Class distribution: [1206 3487]
Original Class weights: {0: np.float64(1.9456882255389718), 1: np.float64(0.6729280183538858)}
Adjusted Class weights: {0: 1.6, 1: 0.8}
time: 47 ms (started: 2026-10-02 21:27:20 +05:30)


In [10]:
from tensorflow.keras.optimizers import Adam

def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 4
        layers.Conv2D(256, (3, 3), padding='same',
              kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),

        layers.GlobalAveragePooling2D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return m

time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


layers.GlobalAveragePooling2D() :-
To reduce parameters and prevent overfitting by summarizing each feature map instead of flattening everything

In [11]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        str(MODELS_DIR / "best_model_class_weights.h5"),
        monitor="val_loss",
        save_best_only=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

time: 0 ns (started: 2026-10-02 21:27:20 +05:30)


In [12]:
model = build_model()
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6950 - loss: 0.8980

147/147 ━━━━━━━━━━━━━━━━━━━━ 361s 2s/step - accuracy: 0.7726 - loss: 0.7829 - val_accuracy: 0.7347 - val_loss: 1.5747 - learning_rate: 1.0000e-04
Epoch 2/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 608s 4s/step - accuracy: 0.8692 - loss: 0.6115 - val_accuracy: 0.7347 - val_loss: 2.7281 - learning_rate: 1.0000e-04
Epoch 3/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 569s 4s/step - accuracy: 0.8922 - loss: 0.5556 - val_accuracy: 0.7347 - val_loss: 2.8967 - learning_rate: 1.0000e-04
Epoch 4/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 513s 3s/step - accuracy: 0.9018 - loss: 0.5199 - val_accuracy: 0.7347 - val_loss: 1.8773 - learning_rate: 3.0000e-05
Epoch 5/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9058 - loss: 0.4929

147/147 ━━━━━━━━━━━━━━━━━━━━ 525s 4s/step - accuracy: 0.8973 - loss: 0.5114 - val_accuracy: 0.7347 - val_loss: 1.3697 - learning_rate: 3.0000e-05
Epoch 6/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9082 - loss: 0.5037

147/147 ━━━━━━━━━━━━━━━━━━━━ 475s 3s/step - accuracy: 0.9060 - loss: 0.5034 - val_accuracy: 0.7551 - val_loss: 0.8779 - learning_rate: 3.0000e-05
Epoch 7/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9054 - loss: 0.5108

147/147 ━━━━━━━━━━━━━━━━━━━━ 584s 4s/step - accuracy: 0.9114 - loss: 0.4866 - val_accuracy: 0.8924 - val_loss: 0.5524 - learning_rate: 3.0000e-05
Epoch 8/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9120 - loss: 0.4673

147/147 ━━━━━━━━━━━━━━━━━━━━ 499s 3s/step - accuracy: 0.9122 - loss: 0.4725 - val_accuracy: 0.8942 - val_loss: 0.5292 - learning_rate: 3.0000e-05
Epoch 9/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9133 - loss: 0.4709

147/147 ━━━━━━━━━━━━━━━━━━━━ 634s 4s/step - accuracy: 0.9143 - loss: 0.4660 - val_accuracy: 0.9035 - val_loss: 0.5020 - learning_rate: 3.0000e-05
Epoch 10/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 496s 3s/step - accuracy: 0.9135 - loss: 0.4586 - val_accuracy: 0.7755 - val_loss: 0.7889 - learning_rate: 3.0000e-05
Epoch 11/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 508s 3s/step - accuracy: 0.9150 - loss: 0.4592 - val_accuracy: 0.7996 - val_loss: 0.7014 - learning_rate: 3.0000e-05
Epoch 12/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 595s 4s/step - accuracy: 0.9195 - loss: 0.4390 - val_accuracy: 0.8720 - val_loss: 0.5446 - learning_rate: 9.0000e-06
Epoch 13/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 553s 4s/step - accuracy: 0.9177 - loss: 0.4460 - val_accuracy: 0.9017 - val_loss: 0.5057 - learning_rate: 9.0000e-06
Epoch 14/25
147/147 ━━━━━━━━━━━━━━━━━━━━ 627s 4s/step - accuracy: 0.9205 - loss: 0.4411 - val_accuracy: 0.8330 - val_loss: 0.6419 - learning_rate: 2.7000e-06
time: 2h 5min 49s (started: 2026-10-02 21:27:20 +05:30)


In [13]:
print("End")

End
time: 15 ms (started: 2026-10-02 23:33:11 +05:30)
